In [68]:
import pandas as pd

In [69]:
df = pd.read_csv('bike_hours_raw.csv')
df.head(10)

,record_id,service_date,station,hour,forecast_temp_c,forecast_rain,departures
0,2072,2026-08-27,Market,20,30.2 C,No,9
1,2595,2026-09-02,Harbor,11,23.9 C,No,6
2,561,2026-08-09,Park,7,27.6 C,No,7
3,3258,2026-09-10,Park,16,25.9 C,No,11
4,4059,2026-09-20,Riverside,19,18.6 C,No,7
5,984,2026-08-14,Park,10,27.2 C,No,4
6,249,2026-08-05,Harbor,17,21.1 C,No,14
7,469,2026-08-08,Market,13,23.7 C,Yes,11
8,3012,2026-09-07,Harbor,8,24.5 C,Yes,7
9,2977,2026-09-07,University,15,24.5 C,Yes,7


Remove exact duplicates

In [70]:
df.duplicated().sum()

np.int64(55)

In [71]:
df.drop_duplicates(inplace=True)
df.duplicated().sum()

np.int64(0)

standardize station names

In [72]:
df["station"].unique()

array(['Market', 'Harbor', 'Park', 'Riverside', 'University', ' harbor ',
       ' park ', ' riverside ', 'Central', ' market ', ' central ',
       ' university '], dtype=object)

In [73]:
df["station"] = df["station"].str.strip().str.capitalize()
df["station"].unique()

array(['Market', 'Harbor', 'Park', 'Riverside', 'University', 'Central'],
      dtype=object)

parse dates and weather fields, and identify invalid hours and departure counts

In [74]:
df["service_date"] = pd.to_datetime(df["service_date"], format="mixed", errors="coerce")
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4704 entries, 0 to 4758
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   record_id        4704 non-null   int64         
 1   service_date     4669 non-null   datetime64[ns]
 2   station          4704 non-null   object        
 3   hour             4704 non-null   int64         
 4   forecast_temp_c  4704 non-null   object        
 5   forecast_rain    4704 non-null   object        
 6   departures       4704 non-null   int64         
dtypes: datetime64[ns](1), int64(3), object(3)
memory usage: 294.0+ KB


In [75]:
df["forecast_rain"]=df["forecast_rain"].str.strip().str.lower().map({"yes":True,"no":False})
df

,record_id,service_date,station,hour,forecast_temp_c,forecast_rain,departures
0,2072,2026-08-27,Market,20,30.2 C,False,9
1,2595,2026-09-02,Harbor,11,23.9 C,False,6
2,561,2026-08-09,Park,7,27.6 C,False,7
3,3258,2026-09-10,Park,16,25.9 C,False,11
4,4059,2026-09-20,Riverside,19,18.6 C,False,7
...,...,...,...,...,...,...,...
4754,2801,2026-09-05,University,7,30.1 C,False,14
4755,1338,2026-08-18,Harbor,14,15.8 C,True,4
4756,407,2026-08-07,Harbor,7,24.6 C,False,13
4757,2192,2026-08-29,Central,14,24.4 C,True,16


In [76]:
df["forecast_temp_c"] = df["forecast_temp_c"] = pd.to_numeric(
    df["forecast_temp_c"].str.replace("C","",regex=True),
    errors="coerce"
)
df["forecast_temp_c"] = df["forecast_temp_c"].round()
df

,record_id,service_date,station,hour,forecast_temp_c,forecast_rain,departures
0,2072,2026-08-27,Market,20,30.0,False,9
1,2595,2026-09-02,Harbor,11,24.0,False,6
2,561,2026-08-09,Park,7,28.0,False,7
3,3258,2026-09-10,Park,16,26.0,False,11
4,4059,2026-09-20,Riverside,19,19.0,False,7
...,...,...,...,...,...,...,...
4754,2801,2026-09-05,University,7,30.0,False,14
4755,1338,2026-08-18,Harbor,14,16.0,True,4
4756,407,2026-08-07,Harbor,7,25.0,False,13
4757,2192,2026-08-29,Central,14,24.0,True,16


In [77]:
df.isnull().sum()

record_id           0
service_date       35
station             0
hour                0
forecast_temp_c    45
forecast_rain       0
departures          0
dtype: int64

Decide what to do with missing forecast temperatures and explain the tradeoff.

Record all applicable rejection reasons for quarantined rows. Save usable rows as bike_hours_clean.csv.

In [78]:
checks = pd.DataFrame({
    "missing_or_invalid_temperatures":(
        df["forecast_temp_c"].isna() | df["forecast_temp_c"].lt(0)
    ),
    "invalid_date":df["service_date"].isna(),
}, index=df.index)

reasons = checks.apply(
    lambda row: ";".join(row.index[row]),
    axis=1
)

rejected = checks.any(axis=1)

bike_hours_quarantine = df.loc[rejected].copy()
bike_hours_quarantine["rejection_reasons"] = reasons.loc[rejected]

bike_hours_clean = df.loc[~rejected].copy()
bike_hours_clean.to_csv("bike_hours_clean.csv",index=False)

bike_hours_quarantine.to_csv(
    "bike_hours_quarantine.csv", index=False
)
